# IOAI — 2025 National Selection Weak Label Cv (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-national-selection-weak-label-cv/data.zip', 'd.zip')
    with zipfile.ZipFile('d.zip') as z: z.extractall('data')
print('데이터 준비:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Weak-Label CV — 약한 라벨로 이미지 분류 (Georgia National Selection 2025)

라벨이 없는 이미지들과 **텍스트 캡션**(`captions.csv`)만 주어진다. 캡션에서 **약한 라벨**을 유도해 8클래스
(airplane/car/cat/dog/flower/fruit/motorbike/person) 이미지 분류기를 **처음부터** 학습한다
(사전학습·CLIP·ViT·ResNet 다운로드 금지). 128×128 RGB. 채점: held-out **accuracy**. 제출:
`submission.json`={이미지경로: 클래스}. T4 ≤1h.

In [ ]:
import pandas as pd, json, os, glob
captions = pd.read_csv("data/captions.csv")      # path, caption (약한 텍스트 라벨)
test = pd.read_csv("data/test_imgs.csv")         # path (라벨 없음)
print("caption imgs", len(captions), "| test imgs", len(test))

In [ ]:
# 베이스라인: 전부 "cat" (starter) — accuracy ≈ 1/8
def create_predictions_json(paths):
    predictions = {p[p.index("test_imgs"):] if "test_imgs" in p else p: "cat" for p in paths}
    with open("submission.json", "w") as f:
        json.dump(predictions, f, indent=4)
    print("saved submission.json", len(predictions))
create_predictions_json(test["path"].tolist())

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)